# Healthcare and Medicine: AI-Powered Disease Prediction and Medical Information Assistant

## Project objective

This notebook demonstrates a complete machine-learning workflow for educational and research purposes:

1. Load multiple healthcare-related datasets independently.
2. Inspect and clean each dataset.
3. Check compatibility before considering any merge.
4. Select a suitable binary classification task.
5. Build leakage-resistant preprocessing pipelines.
6. Train and compare five classification algorithms.
7. Evaluate accuracy, precision, recall, F1-score, confusion matrices, and classification reports.
8. Export the complete pipeline to `disease_model.pkl`.
9. Reload the pipeline and generate an illustrative prediction.

> **Important:** This notebook is not a medical diagnosis system. Predictions must not be used for medical decisions. The datasets may be synthetic or otherwise non-clinical; their original documentation must be checked before making claims about provenance.


## 1. Project Introduction

The supplied repository contains a general disease-symptom dataset, a heart-disease dataset, and a diabetes-related dataset. No verified lung-disease target dataset was supplied in the request. Therefore, the notebook loads all three datasets for inspection and uses the heart-disease dataset as the default binary-classification demonstration when it is available. This prevents incorrectly presenting a heart-disease model as a lung-disease model.

## 2. Importing Libraries

In [ ]:
# Install packages if needed in Google Colab
# !pip -q install pandas numpy scikit-learn seaborn matplotlib joblib requests

import io
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MODEL_FILE = "disease_model.pkl"
TEST_SIZE = 0.20


## 3. Dataset Sources

In [ ]:
DATASET_SOURCES = {
    "general_disease_symptoms": {
        "name": "General Disease-Symptom Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv",
        "target": "Disease",
        "description": "Disease labels with up to twelve symptom columns."
    },
    "heart_disease": {
        "name": "Heart Disease Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/heart_disease.csv",
        "target": "Heart Disease Status",
        "description": "Demographic, lifestyle, laboratory-style, and risk-factor columns."
    },
    "diabetes": {
        "name": "Diabetes Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/diabetes_dataset.csv",
        "target": "Target",
        "description": "Diabetes-related labels and many categorical and numerical attributes."
    }
}

sources_table = pd.DataFrame([
    {
        "dataset_key": key,
        "dataset_name": value["name"],
        "target_column": value["target"],
        "source_url": value["url"]
    }
    for key, value in DATASET_SOURCES.items()
])
display(sources_table)


## 4. Loading Multiple Datasets

In [ ]:
def standardize_column_name(column):
    """Convert a column name into a consistent snake_case format."""
    column = str(column).strip()
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column).strip("_").lower()
    return column

def load_csv_from_url(url, dataset_name, timeout=30):
    """Load a CSV from a URL without crashing the whole notebook if unavailable."""
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        df = pd.read_csv(io.BytesIO(response.content))
        print(f"Loaded: {dataset_name}")
        print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
        return df
    except Exception as error:
        print(f"Could not load {dataset_name}: {error}")
        return None

raw_datasets = {}
for key, info in DATASET_SOURCES.items():
    raw_datasets[key] = load_csv_from_url(info["url"], info["name"])

available_datasets = {
    key: df for key, df in raw_datasets.items() if df is not None
}
print("\nAvailable datasets:", list(available_datasets.keys()))


## 5. Displaying Sample Data

In [ ]:
for key, df in available_datasets.items():
    print("=" * 90)
    print(DATASET_SOURCES[key]["name"])
    print("Source:", DATASET_SOURCES[key]["url"])
    display(df.head(10))


## 6. Dataset Exploration

In [ ]:
for key, df in available_datasets.items():
    print("=" * 90)
    print(DATASET_SOURCES[key]["name"])
    print("Shape:", df.shape)
    print("Column names:")
    print(list(df.columns))
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))
    print("\nInfo:")
    df.info()
    print("\nDescriptive summary:")
    display(df.describe(include="all").transpose().head(50))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))


## 7. Data Cleaning

In [ ]:
def clean_basic_dataframe(df):
    """Perform safe, general cleaning without assuming medical meaning."""
    cleaned = df.copy()

    # Standardize column names.
    cleaned.columns = [standardize_column_name(c) for c in cleaned.columns]

    # Treat common textual missing-value markers as actual missing values.
    missing_markers = ["", " ", "na", "n/a", "null", "none", "missing", "?"]
    cleaned = cleaned.replace(missing_markers, np.nan)

    # Remove exact duplicate rows.
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)
    print(f"Removed {before - len(cleaned)} duplicate rows.")

    # Strip whitespace from object/string columns.
    for col in cleaned.select_dtypes(include=["object", "string"]).columns:
        cleaned[col] = cleaned[col].astype("string").str.strip()

    return cleaned

cleaned_datasets = {}
for key, df in available_datasets.items():
    print("\nCleaning:", DATASET_SOURCES[key]["name"])
    cleaned_datasets[key] = clean_basic_dataframe(df)
    print("Cleaned shape:", cleaned_datasets[key].shape)


## 8. Data Preprocessing and Compatibility Checks

In [ ]:
def find_standardized_target(df, original_target):
    """Find a target column after column-name standardization."""
    target = standardize_column_name(original_target)
    if target in df.columns:
        return target
    return None

for key, df in cleaned_datasets.items():
    target = find_standardized_target(df, DATASET_SOURCES[key]["target"])
    print(f"{key}: expected target={standardize_column_name(DATASET_SOURCES[key]['target'])}, found={target}")
    if target is not None:
        print("Target value counts:")
        display(df[target].value_counts(dropna=False).head(20).to_frame("count"))

print("\nCompatibility note:")
print(
    "The datasets have different targets, feature meanings, units, and label spaces. "
    "They must not be concatenated row-wise or merged by column name without a documented "
    "data-integration design. This notebook therefore analyzes them separately."
)


## 9. Sample Data Demonstration

The following rows are illustrative examples only. They are not genuine medical records and must not be treated as clinical evidence.

In [ ]:
sample_patient_format = pd.DataFrame([
    [45, "Male", "Yes", "Yes", "Yes", "No", "Yes", 1],
    [28, "Female", "No", "No", "No", "No", "No", 0],
    [62, "Male", "Yes", "Yes", "Yes", "Yes", "Yes", 1],
    [35, "Female", "No", "Yes", "No", "No", "Yes", 0],
    [51, "Male", "Yes", "Yes", "Yes", "No", "Yes", 1],
], columns=[
    "Age", "Gender", "Smoking", "Cough", "Breathlessness",
    "Chest_Pain", "Fatigue", "Lung_Disease"
])
display(sample_patient_format)
print("These values are synthetic illustrations, not patient records.")


## 10. Selecting a Demonstration Task and Train-Test Splitting

Because the supplied files do not include a verified lung-disease target, the default task below uses `Heart Disease Status` if it is present and binary. The exported file is consequently a heart-disease demonstration model, not a lung-disease model. To train a true lung-disease model, replace the dataset URL and mapping with a verified lung-disease dataset.

In [ ]:
# Select a safe, available binary target.
TASK_KEY = None
TARGET_COLUMN = None

preferred_tasks = [
    ("heart_disease", "heart_disease_status"),
]

for key, target in preferred_tasks:
    if key in cleaned_datasets and target in cleaned_datasets[key].columns:
        unique_count = cleaned_datasets[key][target].dropna().nunique()
        if unique_count == 2:
            TASK_KEY = key
            TARGET_COLUMN = target
            break

if TASK_KEY is None:
    raise ValueError(
        "No suitable binary demonstration target was found. "
        "Add a verified binary dataset and update the task-selection section."
    )

task_df = cleaned_datasets[TASK_KEY].copy()
task_df = task_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = task_df.drop(columns=[TARGET_COLUMN])
y_raw = task_df[TARGET_COLUMN].astype(str).str.strip()

# Encode the target only; feature preprocessing remains inside the pipeline.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)

print("Selected dataset:", DATASET_SOURCES[TASK_KEY]["name"])
print("Target column:", TARGET_COLUMN)
print("Target classes:", list(target_encoder.classes_))
print("Feature shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 11. Building the Preprocessing Pipeline

In [ ]:
# Identify numerical and categorical columns from the training data only.
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 12. Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Support Vector Machine": SVC(
        probability=True, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

fitted_pipelines = {}
predictions = {}
evaluation_rows = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    fitted_pipelines[model_name] = pipeline
    predictions[model_name] = y_pred

    evaluation_rows[model_name] = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

print("Training complete.")


## 13. Model Evaluation

In [ ]:
for model_name, y_pred in predictions.items():
    print("=" * 90)
    print(model_name)
    print("\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=target_encoder.classes_,
        yticklabels=target_encoder.classes_
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()


## 14. Model Comparison

Accuracy alone is not sufficient for healthcare-related classification. Recall measures how many actual positive cases were detected. A false negative occurs when a positive case is incorrectly predicted as negative; in a screening context, this can be especially important. Precision, recall, F1-score, class-specific metrics, calibration, and the consequences of errors should all be considered.

In [ ]:
comparison_df = pd.DataFrame(list(evaluation_rows.values()))
comparison_df = comparison_df.sort_values(
    by=["recall", "f1_score"], ascending=False
).reset_index(drop=True)

display(comparison_df)

print(
    "The table contains actual results produced on this train/test split. "
    "Do not treat these metrics as clinical validation."
)


## 15. Final Model Selection

In [ ]:
# Educational selection rule:
# prioritize weighted recall, then weighted F1-score.
# This is not a clinical decision rule.
selected_model_name = comparison_df.iloc[0]["model"]
final_pipeline = fitted_pipelines[selected_model_name]

print("Selected model for export:", selected_model_name)
print(
    "Selection was based on the displayed test-set metrics, prioritizing recall and then F1-score. "
    "For real healthcare deployment, use external validation, subgroup analysis, calibration, "
    "and domain-expert review."
)


## 16. Saving the Complete Pipeline

In [ ]:
export_bundle = {
    "pipeline": final_pipeline,
    "target_encoder": target_encoder,
    "feature_columns": list(X.columns),
    "dataset_key": TASK_KEY,
    "target_column": TARGET_COLUMN,
    "selected_model_name": selected_model_name,
    "random_state": RANDOM_STATE,
    "note": (
        "Educational model only. Not a medical diagnosis system. "
        "The pipeline includes imputing, encoding, scaling, and the classifier."
    )
}

joblib.dump(export_bundle, MODEL_FILE)
print(f"Saved model bundle to: {MODEL_FILE}")
print("File exists:", os.path.exists(MODEL_FILE))


## 17. Loading and Testing the Saved Model

In [ ]:
loaded_bundle = joblib.load(MODEL_FILE)
loaded_pipeline = loaded_bundle["pipeline"]
loaded_target_encoder = loaded_bundle["target_encoder"]
expected_features = loaded_bundle["feature_columns"]

# Create a sample record using the first test row so that all required
# feature columns are available. This is only a software test fixture.
sample_record = X_test.iloc[[0]].copy()

loaded_encoded_prediction = loaded_pipeline.predict(sample_record)
loaded_label_prediction = loaded_target_encoder.inverse_transform(
    loaded_encoded_prediction.astype(int)
)

print("Sample input record:")
display(sample_record)
print("Predicted class:", loaded_label_prediction[0])

if hasattr(loaded_pipeline, "predict_proba"):
    probabilities = loaded_pipeline.predict_proba(sample_record)[0]
    probability_table = pd.DataFrame({
        "class": loaded_target_encoder.classes_,
        "probability": probabilities
    })
    display(probability_table)
else:
    print("Probability output is not available for this pipeline.")


## 18. Limitations and Ethical Considerations

- The supplied datasets may be synthetic, simulated, scraped, or otherwise non-clinical. Verify each original source before describing provenance.
- The default demonstration is based on the supplied heart-disease file, not a verified lung-disease dataset.
- A random train/test split is not a substitute for external validation.
- Dataset imbalance, duplicate patterns, label noise, and hidden leakage may distort metrics.
- Accuracy, precision, recall, and F1-score do not establish clinical safety.
- False negatives and false positives have different consequences and should be assessed with domain experts.
- The model may perform differently across demographic or socioeconomic groups.
- Do not use the exported file to diagnose, triage, prescribe, or recommend treatment.
- A future system should include external validation, calibration, uncertainty handling, human review, audit logs, privacy protections, and clear referral guidance.


## 19. Final Conclusion

This notebook creates a reproducible end-to-end machine-learning pipeline, evaluates multiple classifiers, displays real metrics generated from the available data, and exports the complete preprocessing-and-model pipeline as `disease_model.pkl`.

To convert this demonstration into a genuine lung-disease project, add a verified lung-disease dataset with a clearly defined target, update `DATASET_SOURCES`, confirm the feature meanings and label semantics, and rerun the notebook. Do not merge unrelated disease datasets merely because they are all healthcare-related.